# 04 · Pricing Logic Explainer

**Workshop:** AI for Actuaries — From Foundations to AI Agents
**Session / Part:** S2.P2  ·  **Slides:** S2.P2.6–15
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Kasyap
**Date:** 24 July 2026 · Four Points by Sheraton, Whitefield, Bangalore
**Model:** `gemini-3.1-flash-lite` (pinned)  ·  **License:** CC BY-NC 4.0

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohanyashraj/ifoa-workshop/blob/main/notebooks/04_pricing_logic_explainer.ipynb)

## What this notebook does
The signature build: three tools, a contract-grade system prompt, a live hallucination (the air-filter discount), and the guardrail that kills it. Then five-line ports to Health and Life.

*All data is hypothetical — ABC Insurer is a fictional entity for teaching only.
The story: Priya Nair (pricing, ABC General) must explain the price of policy
**ABC-MOT-047231** — a 7-year-old SUV, Tier-2, 35% NCB — so her chief actuary
**Arjun Mehta** can sign it.*

In [ ]:
%pip install -q agno google-genai

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    assert "GOOGLE_API_KEY" in os.environ
from google import genai
client = genai.Client()
MODEL = "gemini-3.1-flash-lite"

## 1. Tool 1 — load_rating_table (deterministic, auditable)

In [ ]:
RATING_TABLE = {                       # ABC Motor 2024 — illustrative
    "base_frequency": 0.082,
    "vehicle_age": {0: 0.85, 7: 1.18, 12: 1.35},
    "ncb_pct":     {0: 1.00, 35: 0.82, 50: 0.72},
    "region":      {"Tier1": 0.95, "Tier2": 1.05, "Tier3": 1.12},
}

def load_rating_table() -> dict:
    """Return ABC Motor's official rating table: base frequency + relativities.
    Deterministic — no LLM inside."""
    return RATING_TABLE

## 2. Tool 2 — explain_factor (LLM only where language is the job)

In [ ]:
def explain_factor(factor: str, value: str, relativity: float) -> str:
    """Explain ONE rating factor's effect in one plain-English sentence.
    Python supplies the number; the LLM only phrases it."""
    prompt = (f"In ONE sentence for a customer, explain that {factor}={value} "
              f"multiplies the premium by {relativity}. Add no new numbers.")
    return client.models.generate_content(model=MODEL, contents=prompt).text.strip()

## 3. Tool 3 + the contract — generate_doc and the system prompt

In [ ]:
def generate_doc(policy_id: str, lines: list) -> str:
    """Assemble the memo — pure string join, NO LLM. Same headers every time."""
    body = "\n".join(f"- {x}" for x in lines)
    return f"PRICING MEMO · {policy_id}\n{'='*32}\n{body}"

SYSTEM = ("You explain ABC General motor prices to underwriters. "
          "Use ONLY load_rating_table for numbers. NEVER invent a factor. "
          "Keep the memo under 180 words.")

## 4. Register, invoke — first run reconciles

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini

agent = Agent(model=Gemini(id=MODEL),
              tools=[load_rating_table, explain_factor, generate_doc],
              system_message=SYSTEM, show_tool_calls=True)

agent.print_response("Explain the price for policy ABC-MOT-047231: "
                     "a 7-year-old SUV, Tier-2, 35% NCB.")

## 5. The failure — hallucination on cue ⚠️
The system prompt says *never invent a factor*. Watch it anyway.

In [ ]:
agent.print_response("How does the air-filter discount apply to policy ABC-MOT-047231?")
# Expect: it explains the real factors, then confidently invents an 'air-filter discount'.

## 6. The guardrail — a tool that gatekeeps the tools

In [ ]:
def check_factor_in_table(factor: str) -> dict:
    """Deterministic gate: does this factor exist in the rating table?"""
    tbl = load_rating_table()
    return {"valid": factor in tbl, "valid_factors": list(tbl)}

SYSTEM_GUARDED = SYSTEM + (" Before calling explain_factor, ALWAYS call "
    "check_factor_in_table. If invalid, refuse and list the valid factors. Never invent.")

guarded = Agent(model=Gemini(id=MODEL),
                tools=[load_rating_table, explain_factor, generate_doc, check_factor_in_table],
                system_message=SYSTEM_GUARDED, show_tool_calls=True)

## 7. Second run — the gate holds

In [ ]:
guarded.print_response("How does the air-filter discount apply to policy ABC-MOT-047231?")
# Expect: check_factor_in_table('air_filter') -> False -> polite refusal listing real factors.

## TODO — the five-line port
Port the agent to **Health** (Dr Ananya Iyer): swap `RATING_TABLE` for a severity table by procedure and member age, and swap the persona in `SYSTEM`. `explain_factor`, `generate_doc` and the guardrail stay **unchanged**.

```python
# HEALTH_TABLE = {...}
# SYSTEM_H = SYSTEM_GUARDED.replace('motor prices', 'hospitalisation costs')
# ...
```

## Faster with Colab's own Gemini
You don't need an external IDE. In Colab, use **Generate** (describe a function in a comment), inline **autocomplete**, the **explain-error** helper on a traceback, and the **chat sidebar** ('refactor this cell into a tool with a docstring'). Same reasoner, pointed at your code — you stay in the review seat.

## Wrap-up
*Demonstrated: a governed agent in ~60 lines — it worked, it lied, and a unit-testable gate made it stop. This is the memo Arjun signs.*

**Next:** `05_actuarial_analyst_agent.ipynb`.